# 🤖 NewsBot Intelligence System
## ITAI 2373 — Mid-Term Project
### Student Name: *(Enter your full name here)*
### Course: ITAI 2373 — Natural Language Processing
### Date: April 2025

---

## 📋 Project Overview
This NewsBot Intelligence System integrates all NLP techniques from Modules 1–8 into a single end-to-end pipeline. I designed, implemented, and analyzed every component independently.

| Module | Topic | Status |
|--------|-------|--------|
| 1 | Real-World NLP Application Context | ✅ |
| 2 | Text Preprocessing Pipeline | ✅ |
| 3 | TF-IDF Feature Extraction | ✅ |
| 4 | Part-of-Speech Pattern Analysis | ✅ |
| 5 | Syntax Parsing & Semantic Analysis | ✅ |
| 6 | Sentiment & Emotion Analysis | ✅ |
| 7 | Multi-Class Text Classification | ✅ |
| 8 | Named Entity Recognition | ✅ |

**Dataset**: BBC News Classification Dataset (Kaggle)  
**Platform**: Google Colab (free tier)


---
# 📦 Environment Setup
Run this cell **first** every time you open the notebook in Google Colab.

In [ ]:
# ============================================================
# CELL 1: Install all required libraries
# ============================================================
!pip install nltk spacy scikit-learn pandas matplotlib seaborn wordcloud textblob --quiet
!python -m spacy download en_core_web_sm --quiet

import nltk
for resource in ['punkt', 'punkt_tab', 'stopwords', 'averaged_perceptron_tagger',
                 'averaged_perceptron_tagger_eng', 'wordnet', 'vader_lexicon',
                 'maxent_ne_chunker', 'words']:
    nltk.download(resource, quiet=True)

print("✅ All libraries installed and NLTK resources downloaded!")


In [ ]:
# ============================================================
# CELL 2: Import all libraries
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
import re
import warnings
warnings.filterwarnings('ignore')

# NLTK
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Scikit-learn
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, accuracy_score,
                              confusion_matrix, ConfusionMatrixDisplay)
from sklearn.pipeline import Pipeline

# spaCy
import spacy
nlp = spacy.load('en_core_web_sm')

# TextBlob
from textblob import TextBlob

# Visualization
from wordcloud import WordCloud

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ All imports successful!")
print("🤖 NewsBot Intelligence System ready!")


---
# 📰 Module 1: Real-World NLP Application Context

## Business Case
News organizations, financial institutions, and PR firms face a common challenge: **information overload**. Thousands of news articles are published every hour, making manual monitoring impossible. The **NewsBot Intelligence System** solves this by automatically classifying, analyzing, and extracting insights from news at scale.

## Target Users & Value
| User | Problem Solved | Business Value |
|------|---------------|----------------|
| Media companies | Track competitor coverage | Save 10+ analyst hours/day |
| Financial firms | Monitor market-moving news | Faster trading signals |
| PR agencies | Monitor brand mentions + sentiment | Real-time reputation management |
| Researchers | Analyze opinion trends at scale | Large-scale study without manual reading |

## System Architecture
```
Raw News Article
      ↓
[Module 2] Text Preprocessing
      ↓
[Module 3] TF-IDF Feature Extraction ──→ Category Keywords
      ↓
[Module 4] POS Tagging ──────────────→ Writing Style Analysis
      ↓
[Module 5] Syntax Parsing ───────────→ Structural Patterns
      ↓
[Module 6] Sentiment Analysis ───────→ Emotional Tone
      ↓
[Module 7] Classification ───────────→ Category Label
      ↓
[Module 8] Named Entity Recognition ─→ People / Orgs / Places
      ↓
Business Intelligence Dashboard
```


In [ ]:
# ============================================================
# CELL 3: Dataset Loading
# ============================================================
# HOW TO USE YOUR REAL BBC DATA:
# Option A — Direct upload to Colab:
#   1. Go to https://www.kaggle.com/competitions/learn-ai-bbc/data
#   2. Download train.csv
#   3. Upload via Colab folder icon (left sidebar)
#   4. Uncomment the line below:
# df = pd.read_csv('train.csv')

# Option B — Kaggle API:
# !pip install kaggle --quiet
# from google.colab import files
# uploaded = files.upload()  # Upload kaggle.json when prompted
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle competitions download -c learn-ai-bbc
# !unzip learn-ai-bbc.zip
# df = pd.read_csv('train.csv')

# ---- DEMO DATA (runs immediately, replace with real data above) ----
demo_data = {
    'text': [
        "The government announced sweeping new economic policies aimed at reducing unemployment and stimulating GDP growth through infrastructure spending.",
        "Scientists at Johns Hopkins University discovered a breakthrough treatment for Type 2 diabetes that could help millions of patients worldwide.",
        "Manchester United won the Premier League championship after a thrilling penalty shootout victory against Liverpool in the final match.",
        "Apple unveiled its latest iPhone model featuring revolutionary on-device AI processing capabilities and a significantly improved battery life.",
        "Hollywood actress Cate Blanchett received her third Oscar nomination for her powerful performance in the acclaimed biographical drama.",
        "Global stock markets surged on Friday as investors reacted positively to the Federal Reserve decision to hold interest rates steady.",
        "The prime minister called an emergency cabinet meeting to address the escalating immigration crisis at the southern border.",
        "Researchers at MIT developed a machine learning algorithm that detects early-stage cancer from routine blood samples with 94% accuracy.",
        "NBA star LeBron James signed a record-breaking two-year contract extension worth 97 million dollars with the Los Angeles Lakers.",
        "Netflix released its most expensive original production — a historical epic featuring stunning visual effects and an international all-star cast.",
        "Parliament voted 312 to 198 to approve the controversial austerity budget that includes significant cuts to public services.",
        "A Phase 3 clinical trial shows a new mRNA vaccine provides strong protection against all known influenza strains with minimal side effects.",
        "Serena Williams announced her return to professional tennis following injury recovery, aiming to compete at the Australian Open.",
        "Amazon Web Services announced a 10 billion dollar expansion of its cloud computing infrastructure across three new data center regions.",
        "Oscar-winning director Martin Scorsese confirmed his retirement from feature filmmaking after a celebrated 45-year career.",
        "The European Central Bank raised its benchmark interest rate by 25 basis points for the fourth consecutive quarter to fight inflation.",
        "Opposition party leaders held a press conference criticizing the government over its failure to address the worsening housing affordability crisis.",
        "Cardiologists published findings showing a Mediterranean diet reduces the risk of major cardiovascular events by 38% over a 10-year period.",
        "Jamaican sprinter Oblique Seville broke the 100-meter world record at the World Athletics Championships in Budapest.",
        "Meta faced renewed congressional scrutiny over algorithmic amplification of misinformation and its effects on teenage mental health.",
        "Treasury Secretary announced a new public-private partnership to fund renewable energy infrastructure worth 50 billion dollars.",
        "Health officials issued guidance recommending annual booster shots for COVID-19 as new variants continue to emerge globally.",
        "Chelsea Football Club signed a Brazilian midfielder for a reported transfer fee of 120 million euros breaking the club record.",
        "Google DeepMind published research demonstrating that its Gemini model achieves human-level performance on standardized medical licensing exams.",
        "The Cannes Film Festival awarded its Palme d'Or to a French drama exploring immigration and identity in contemporary Europe.",
    ],
    'category': [
        'politics','health','sport','tech','entertainment',
        'business','politics','health','sport','entertainment',
        'politics','health','sport','tech','entertainment',
        'business','politics','health','sport','tech',
        'business','health','sport','tech','entertainment',
    ]
}

df = pd.DataFrame(demo_data)

# Standardize column names
if 'text' in df.columns and 'content' not in df.columns:
    df = df.rename(columns={'text': 'content'})
if 'label' in df.columns:
    df = df.rename(columns={'label': 'category'})

# Sample if too large
if len(df) > 2000:
    df = df.sample(n=2000, random_state=42).reset_index(drop=True)

print("=" * 55)
print("📊 DATASET LOADED")
print("=" * 55)
print(f"Articles: {len(df)}  |  Categories: {df['category'].nunique()}")
print(f"Columns:  {df.columns.tolist()}")
print()
print("Category Distribution:")
for cat, count in df['category'].value_counts().items():
    bar = '█' * count
    print(f"  {cat:<15} {count:>3}  {bar}")


---
# 🧹 Module 2: Text Preprocessing Pipeline

Before any analysis, raw text must be cleaned and normalized. My pipeline applies five sequential steps: lowercasing, URL/noise removal, tokenization, stop word removal, and lemmatization.

In [ ]:
# ============================================================
# CELL 4: Preprocessing Pipeline
# ============================================================
stop_words_set = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text, remove_stopwords=True, lemmatize=True):
    """
    Full text preprocessing pipeline.
    Steps: lowercase → remove noise → tokenize → stopwords → lemmatize
    """
    if not isinstance(text, str):
        return []
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)          # Remove URLs
    text = re.sub(r'\S+@\S+', '', text)                    # Remove emails
    text = re.sub(r'\d+', '', text)                         # Remove numbers
    text = re.sub(r'[^\w\s]', '', text)                    # Remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()                # Normalize spaces
    tokens = word_tokenize(text)
    if remove_stopwords:
        tokens = [t for t in tokens if t not in stop_words_set and len(t) > 2]
    if lemmatize:
        tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return tokens

# Apply to all articles
print("⏳ Preprocessing all articles...")
df['tokens']         = df['content'].apply(preprocess_text)
df['processed_text'] = df['tokens'].apply(lambda x: ' '.join(x))
df['token_count']    = df['tokens'].apply(len)
df['char_count']     = df['content'].apply(len)
df['sentence_count'] = df['content'].apply(lambda x: len(sent_tokenize(x)) if isinstance(x, str) else 0)

print("✅ Preprocessing complete!")
print()
print("Before vs After Preprocessing:")
print(f"  Original:  '{df['content'].iloc[0][:80]}...'")
print(f"  Processed: '{df['processed_text'].iloc[0][:80]}...'")
print()
stats = df.groupby('category').agg(
    Articles=('content','count'),
    Avg_Tokens=('token_count','mean'),
    Avg_Sentences=('sentence_count','mean')
).round(1)
print("Stats per category:")
print(stats)


In [ ]:
# ============================================================
# CELL 5: Preprocessing Visualization
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Article count
cat_counts = df['category'].value_counts()
axes[0].bar(cat_counts.index, cat_counts.values,
            color=sns.color_palette("husl", len(cat_counts)))
axes[0].set_title('Articles per Category', fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

# Avg token count
avg_tok = df.groupby('category')['token_count'].mean().sort_values(ascending=False)
axes[1].bar(avg_tok.index, avg_tok.values,
            color=sns.color_palette("Set2", len(avg_tok)))
axes[1].set_title('Avg Token Count per Category', fontweight='bold')
axes[1].set_ylabel('Tokens')
axes[1].tick_params(axis='x', rotation=30)

# Token distribution
df['token_count'].hist(bins=20, ax=axes[2], color='steelblue', edgecolor='white')
axes[2].set_title('Token Count Distribution', fontweight='bold')
axes[2].set_xlabel('Tokens per Article')
axes[2].set_ylabel('Frequency')

plt.suptitle('Module 2: Preprocessing Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print("✅ Module 2 visualization complete!")


---
# 📊 Module 3: TF-IDF Feature Extraction and Analysis

TF-IDF reveals which words are most distinctive per category. High TF-IDF scores indicate words that appear frequently in a category but rarely across all articles — the true signal words.

In [ ]:
# ============================================================
# CELL 6: TF-IDF Vectorization
# ============================================================
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=1,
    max_df=0.90,
    ngram_range=(1, 2),
    stop_words='english'
)

X_tfidf = tfidf_vectorizer.fit_transform(df['processed_text'])
feature_names = tfidf_vectorizer.get_feature_names_out()

print("=" * 50)
print("📊 TF-IDF RESULTS")
print("=" * 50)
print(f"Matrix:     {X_tfidf.shape[0]} articles × {X_tfidf.shape[1]} features")
sparsity = (1 - X_tfidf.nnz / (X_tfidf.shape[0] * X_tfidf.shape[1])) * 100
print(f"Sparsity:   {sparsity:.1f}%")

print("\n🏆 Top TF-IDF Terms per Category:")
print("-" * 45)
for category in sorted(df['category'].unique()):
    cat_mask = df['category'] == category
    cat_tfidf = X_tfidf[cat_mask].toarray().mean(axis=0)
    top_idx = cat_tfidf.argsort()[-8:][::-1]
    top_terms = [feature_names[i] for i in top_idx]
    print(f"  {category.upper():<15}: {', '.join(top_terms)}")


In [ ]:
# ============================================================
# CELL 7: TF-IDF Word Clouds + Bar Charts
# ============================================================
categories = sorted(df['category'].unique())
n = len(categories)
fig, axes = plt.subplots(2, n, figsize=(n * 4, 9))
if n == 1:
    axes = axes.reshape(2, 1)

cmaps = ['Blues','Greens','Reds','Purples','Oranges','YlOrBr','PuBu','BuGn']

for idx, cat in enumerate(categories):
    cmap = cmaps[idx % len(cmaps)]
    cat_text = ' '.join(df[df['category'] == cat]['processed_text'])

    # Word cloud
    if cat_text.strip():
        wc = WordCloud(width=380, height=220, background_color='white',
                       colormap=cmap, max_words=40).generate(cat_text)
        axes[0][idx].imshow(wc, interpolation='bilinear')
    axes[0][idx].axis('off')
    axes[0][idx].set_title(f'{cat.upper()}', fontsize=11, fontweight='bold')

    # Top terms bar
    cat_mask = df['category'] == cat
    cat_tfidf_mean = X_tfidf[cat_mask].toarray().mean(axis=0)
    top_n = 7
    top_idx = cat_tfidf_mean.argsort()[-top_n:][::-1]
    terms  = [feature_names[i] for i in top_idx]
    scores = [cat_tfidf_mean[i] for i in top_idx]
    cmap_obj = plt.get_cmap(cmap)
    bar_colors = [cmap_obj(0.35 + 0.5*i/top_n) for i in range(top_n)]
    axes[1][idx].barh(terms[::-1], scores[::-1], color=bar_colors[::-1])
    axes[1][idx].set_title('Top TF-IDF Terms', fontsize=9)
    axes[1][idx].tick_params(axis='y', labelsize=8)

plt.suptitle('Module 3: TF-IDF Analysis — Category Vocabulary', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()
print("✅ Module 3 visualization complete!")


---
# 🏷️ Module 4: Part-of-Speech Pattern Analysis

Different news categories have distinct grammatical fingerprints. Sports articles use more action verbs. Business articles are noun-heavy with numbers. Political articles use many proper nouns (names of people and places). POS analysis reveals these patterns.

In [ ]:
# ============================================================
# CELL 8: POS Tagging
# ============================================================
def get_pos_distribution(text):
    """Return % breakdown of POS categories in a text."""
    if not isinstance(text, str) or not text.strip():
        return {}
    try:
        tokens = word_tokenize(text[:600])
        tags   = pos_tag(tokens)
        pos_map = {
            'NN':'Noun','NNS':'Noun','NNP':'Proper Noun','NNPS':'Proper Noun',
            'VB':'Verb','VBD':'Verb','VBG':'Verb','VBN':'Verb','VBP':'Verb','VBZ':'Verb',
            'JJ':'Adjective','JJR':'Adjective','JJS':'Adjective',
            'RB':'Adverb','RBR':'Adverb','RBS':'Adverb',
            'CD':'Number','IN':'Preposition','DT':'Determiner'
        }
        counts = Counter(pos_map.get(tag,'Other') for _, tag in tags)
        total  = sum(counts.values())
        return {k: v/total*100 for k, v in counts.items()} if total else {}
    except:
        return {}

print("⏳ Running POS tagging...")
df['pos_dist'] = df['content'].apply(get_pos_distribution)
print("✅ POS tagging complete!")

pos_categories = ['Noun','Proper Noun','Verb','Adjective','Adverb','Number']
pos_by_cat = {}
for cat in df['category'].unique():
    sub = df[df['category'] == cat]
    pos_by_cat[cat] = {
        pos: np.mean([d.get(pos,0) for d in sub['pos_dist'] if d])
        for pos in pos_categories
    }

pos_summary = pd.DataFrame(pos_by_cat).T.round(2)
print("\n📊 POS Distribution by Category (%):")
print(pos_summary)

print("\n💡 Key POS Insights:")
for pos in ['Proper Noun','Verb','Number']:
    if pos in pos_summary.columns and pos_summary[pos].max() > 0:
        top_cat = pos_summary[pos].idxmax()
        print(f"  Most {pos}s → {top_cat.upper()}")


In [ ]:
# ============================================================
# CELL 9: POS Visualization
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap
if not pos_summary.empty:
    sns.heatmap(pos_summary, annot=True, fmt='.1f', cmap='YlOrRd',
                ax=axes[0], linewidths=0.5, cbar_kws={'label': '% of tokens'})
    axes[0].set_title('POS Distribution Heatmap', fontsize=13, fontweight='bold')
    axes[0].tick_params(axis='x', rotation=30)

# Grouped bar
x = np.arange(len(pos_summary.index))
width = 0.13
pos_cols = [c for c in pos_categories if c in pos_summary.columns]
colors   = sns.color_palette("husl", len(pos_cols))
for i, pos in enumerate(pos_cols):
    axes[1].bar(x + i*width, pos_summary[pos], width,
                label=pos, color=colors[i], alpha=0.85)
axes[1].set_xticks(x + width * len(pos_cols)/2)
axes[1].set_xticklabels(pos_summary.index, rotation=30)
axes[1].set_ylabel('Avg % of Tokens')
axes[1].set_title('POS Comparison Across Categories', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=8)

plt.suptitle('Module 4: Part-of-Speech Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print("✅ Module 4 complete!")


---
# 🔗 Module 5: Syntax Parsing and Semantic Analysis

Using spaCy's dependency parser to extract syntactic relationships. This reveals how sentences are constructed and identifies Subject–Verb–Object triples that capture the core meaning of each sentence.

In [ ]:
# ============================================================
# CELL 10: Dependency Parsing
# ============================================================
def extract_syntactic_features(text, max_chars=600):
    """Extract syntactic features: sentence length, noun chunks, SVO triples."""
    if not isinstance(text, str):
        return {}
    try:
        doc = nlp(text[:max_chars])
        sents = list(doc.sents)
        avg_len = np.mean([len(s) for s in sents]) if sents else 0
        n_chunks = len(list(doc.noun_chunks))
        n_subj = sum(1 for t in doc if 'subj' in t.dep_)
        n_obj  = sum(1 for t in doc if 'obj'  in t.dep_)
        svo = []
        for t in doc:
            if t.dep_ == 'ROOT' and t.pos_ == 'VERB':
                subj = [w.text for w in t.lefts  if 'subj' in w.dep_]
                obj  = [w.text for w in t.rights if 'obj'  in w.dep_]
                if subj and obj:
                    svo.append((subj[0], t.text, obj[0]))
        return {'avg_sentence_length': avg_len, 'n_noun_chunks': n_chunks,
                'n_subjects': n_subj, 'n_objects': n_obj, 'svo_triples': svo}
    except:
        return {}

print("⏳ Running dependency parsing...")
df['syntax'] = df['content'].apply(extract_syntactic_features)
for feat in ['avg_sentence_length','n_noun_chunks','n_subjects','n_objects']:
    df[feat] = df['syntax'].apply(lambda x: x.get(feat,0) if isinstance(x,dict) else 0)

print("✅ Syntax parsing complete!")

syntax_by_cat = df.groupby('category')[
    ['avg_sentence_length','n_noun_chunks','n_subjects','n_objects']
].mean().round(2)
print("\n📊 Syntactic Features by Category:")
print(syntax_by_cat)

print("\n🔗 Sample Subject-Verb-Object Triples:")
for _, row in df.head(8).iterrows():
    triples = row['syntax'].get('svo_triples',[]) if isinstance(row['syntax'],dict) else []
    if triples:
        print(f"  [{row['category']:<13}] {triples[0]}")


In [ ]:
# ============================================================
# CELL 11: Syntax Visualization
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Normalized heatmap
syn_cols = ['avg_sentence_length','n_noun_chunks','n_subjects','n_objects']
syn_avail = [c for c in syn_cols if c in syntax_by_cat.columns]
if syn_avail:
    norm = (syntax_by_cat[syn_avail] - syntax_by_cat[syn_avail].min()) /            (syntax_by_cat[syn_avail].max() - syntax_by_cat[syn_avail].min() + 1e-6)
    sns.heatmap(norm, annot=syntax_by_cat[syn_avail], fmt='.1f', cmap='Blues',
                ax=axes[0], linewidths=0.5)
    axes[0].set_title('Syntactic Complexity by Category\n(normalized, raw values shown)',
                      fontsize=12, fontweight='bold')
    axes[0].tick_params(axis='x', rotation=30)

# Sentence length bar
if 'avg_sentence_length' in df.columns:
    means = df.groupby('category')['avg_sentence_length'].mean().sort_values(ascending=False)
    axes[1].bar(means.index, means.values,
                color=sns.color_palette("Set2", len(means)))
    axes[1].set_title('Avg Sentence Length by Category', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Tokens per Sentence')
    axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('Module 5: Syntactic Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print("✅ Module 5 complete!")


---
# 😊 Module 6: Sentiment and Emotion Analysis

I apply VADER (rule-based, designed for news/social text) and TextBlob (ML-based) in parallel. Together they give compound sentiment score, positive/negative/neutral breakdown, and subjectivity score.

In [ ]:
# ============================================================
# CELL 12: Sentiment Analysis
# ============================================================
sia = SentimentIntensityAnalyzer()

def analyze_sentiment(text):
    """VADER + TextBlob dual sentiment analysis."""
    if not isinstance(text, str):
        return {'compound':0,'pos':0,'neg':0,'neu':1,'label':'neutral','subjectivity':0}
    scores = sia.polarity_scores(text)
    c = scores['compound']
    label = 'positive' if c >= 0.05 else ('negative' if c <= -0.05 else 'neutral')
    blob  = TextBlob(text)
    return {'compound':c, 'pos':scores['pos'], 'neg':scores['neg'],
            'neu':scores['neu'], 'label':label,
            'subjectivity':blob.sentiment.subjectivity,
            'polarity':blob.sentiment.polarity}

print("⏳ Running sentiment analysis...")
df['sentiment']          = df['content'].apply(analyze_sentiment)
df['sentiment_label']    = df['sentiment'].apply(lambda x: x['label'])
df['sentiment_compound'] = df['sentiment'].apply(lambda x: x['compound'])
df['sentiment_pos']      = df['sentiment'].apply(lambda x: x['pos'])
df['sentiment_neg']      = df['sentiment'].apply(lambda x: x['neg'])
df['subjectivity']       = df['sentiment'].apply(lambda x: x['subjectivity'])
print("✅ Sentiment analysis complete!")

print("\nOverall Sentiment Distribution:")
for label, count in df['sentiment_label'].value_counts().items():
    pct = count/len(df)*100
    bar = '█' * int(pct/3)
    print(f"  {label:<10} {count:>3} articles ({pct:.0f}%)  {bar}")

print("\nAvg Compound Score by Category:")
for cat, score in df.groupby('category')['sentiment_compound'].mean().sort_values(ascending=False).items():
    sign = '+' if score >= 0 else ''
    bar  = '▓' * int(abs(score)*15)
    print(f"  {cat:<15} {sign}{score:.3f}  {bar}")


In [ ]:
# ============================================================
# CELL 13: Sentiment Visualization
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Stacked sentiment %
sent_dist = df.groupby(['category','sentiment_label']).size().unstack(fill_value=0)
sent_pct  = sent_dist.div(sent_dist.sum(axis=1), axis=0) * 100
bottom = np.zeros(len(sent_pct))
colors_s = {'positive':'#2ecc71','neutral':'#95a5a6','negative':'#e74c3c'}
for s in ['positive','neutral','negative']:
    if s in sent_pct.columns:
        axes[0][0].bar(sent_pct.index, sent_pct[s], bottom=bottom,
                       label=s.capitalize(), color=colors_s[s], alpha=0.85)
        bottom += sent_pct[s].values
axes[0][0].set_title('Sentiment Distribution by Category (%)', fontsize=12, fontweight='bold')
axes[0][0].set_ylabel('Percentage')
axes[0][0].legend()
axes[0][0].tick_params(axis='x', rotation=30)

# Boxplot compound scores
cat_order = df.groupby('category')['sentiment_compound'].median().sort_values(ascending=False).index.tolist()
data_bp   = [df[df['category']==c]['sentiment_compound'].values for c in cat_order]
bp = axes[0][1].boxplot(data_bp, patch_artist=True)
palette = sns.color_palette("husl", len(cat_order))
for patch, color in zip(bp['boxes'], palette):
    patch.set_facecolor(color); patch.set_alpha(0.7)
axes[0][1].set_xticklabels(cat_order, rotation=30)
axes[0][1].axhline(0, color='red', linestyle='--', linewidth=1)
axes[0][1].set_title('Sentiment Score Distribution', fontsize=12, fontweight='bold')
axes[0][1].set_ylabel('VADER Compound Score')

# Subjectivity vs Compound scatter
pal = dict(zip(df['category'].unique(), sns.color_palette("husl", df['category'].nunique())))
for cat in df['category'].unique():
    sub = df[df['category']==cat]
    axes[1][0].scatter(sub['subjectivity'], sub['sentiment_compound'],
                       label=cat, alpha=0.6, s=45, color=pal[cat])
axes[1][0].axhline(0, color='gray', linestyle='--', linewidth=0.8)
axes[1][0].axvline(0.5, color='gray', linestyle='--', linewidth=0.8)
axes[1][0].set_xlabel('Subjectivity'); axes[1][0].set_ylabel('Compound Score')
axes[1][0].set_title('Subjectivity vs Sentiment', fontsize=12, fontweight='bold')
axes[1][0].legend(fontsize=8)

# Pos vs Neg bars
means = df.groupby('category')[['sentiment_pos','sentiment_neg']].mean()
x = np.arange(len(means)); w = 0.35
axes[1][1].bar(x-w/2, means['sentiment_pos'], w, label='Positive', color='#2ecc71', alpha=0.8)
axes[1][1].bar(x+w/2, means['sentiment_neg'], w, label='Negative', color='#e74c3c', alpha=0.8)
axes[1][1].set_xticks(x); axes[1][1].set_xticklabels(means.index, rotation=30)
axes[1][1].set_title('Positive vs Negative Tone', fontsize=12, fontweight='bold')
axes[1][1].set_ylabel('Avg VADER Score'); axes[1][1].legend()

plt.suptitle('Module 6: Sentiment & Emotion Analysis', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()
print("✅ Module 6 complete!")


---
# 🤖 Module 7: Multi-Class Text Classification

I train and compare four algorithms — Naive Bayes, Logistic Regression, Linear SVM, and Random Forest — using TF-IDF features. Each is evaluated on test accuracy, cross-validation score, and per-category F1-score.

In [ ]:
# ============================================================
# CELL 14: Train & Evaluate 4 Classifiers
# ============================================================
X = df['processed_text']
y = df['category']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,
    stratify=y if y.value_counts().min() >= 2 else None
)
print(f"Train: {len(X_train)}  |  Test: {len(X_test)}")

classifiers = {
    'Naive Bayes': Pipeline([
        ('tfidf', TfidfVectorizer(max_features=3000, ngram_range=(1,2), stop_words='english')),
        ('clf',   MultinomialNB(alpha=0.1))
    ]),
    'Logistic Regression': Pipeline([
        ('tfidf', TfidfVectorizer(max_features=3000, ngram_range=(1,2), stop_words='english')),
        ('clf',   LogisticRegression(max_iter=1000, C=1.0, random_state=42))
    ]),
    'Linear SVM': Pipeline([
        ('tfidf', TfidfVectorizer(max_features=3000, ngram_range=(1,2), stop_words='english')),
        ('clf',   LinearSVC(max_iter=2000, C=1.0, random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('tfidf', TfidfVectorizer(max_features=2000, ngram_range=(1,1), stop_words='english')),
        ('clf',   RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
    ])
}

results, predictions = {}, {}
print("\n⏳ Training classifiers...")
print("-" * 55)
cv_folds = min(5, y.value_counts().min())

for name, pipe in classifiers.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    acc    = accuracy_score(y_test, y_pred)
    cv     = cross_val_score(pipe, X, y, cv=cv_folds, scoring='accuracy')
    results[name]     = {'accuracy': acc, 'cv_mean': cv.mean(), 'cv_std': cv.std(),
                         'report': classification_report(y_test, y_pred, output_dict=True)}
    predictions[name] = y_pred
    print(f"  {name:<22} Test: {acc:.3f}  CV: {cv.mean():.3f}±{cv.std():.3f}")

best_model_name = max(results, key=lambda n: results[n]['accuracy'])
print(f"\n🏆 Best Model: {best_model_name} ({results[best_model_name]['accuracy']:.1%})")


In [ ]:
# ============================================================
# CELL 15: Classification Visualization
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# Accuracy comparison
names = list(results.keys())
accs  = [results[n]['accuracy']  for n in names]
cvs   = [results[n]['cv_mean']   for n in names]
stds  = [results[n]['cv_std']    for n in names]
x = np.arange(len(names)); w = 0.35
b1 = axes[0][0].bar(x-w/2, accs, w, label='Test Accuracy', color='steelblue', alpha=0.85)
b2 = axes[0][0].bar(x+w/2, cvs,  w, yerr=stds, capsize=4, label='CV Accuracy', color='coral', alpha=0.85)
for bar, a in zip(b1, accs):
    axes[0][0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                    f'{a:.3f}', ha='center', fontsize=9, fontweight='bold')
axes[0][0].set_xticks(x); axes[0][0].set_xticklabels(names, rotation=15)
axes[0][0].set_ylim(0,1.1); axes[0][0].set_ylabel('Accuracy')
axes[0][0].set_title('Classifier Accuracy Comparison', fontsize=13, fontweight='bold')
axes[0][0].legend(); axes[0][0].axhline(0.5, color='gray', linestyle='--', linewidth=0.8)

# Per-category F1 for best model
rpt  = results[best_model_name]['report']
cats = [k for k in rpt if k not in ['accuracy','macro avg','weighted avg']]
f1s  = [rpt[c]['f1-score']  for c in cats]
prs  = [rpt[c]['precision'] for c in cats]
rcs  = [rpt[c]['recall']    for c in cats]
x2   = np.arange(len(cats)); w2 = 0.25
axes[0][1].bar(x2-w2, prs, w2, label='Precision', color='#3498db', alpha=0.8)
axes[0][1].bar(x2,    f1s, w2, label='F1',        color='#2ecc71', alpha=0.8)
axes[0][1].bar(x2+w2, rcs, w2, label='Recall',    color='#e74c3c', alpha=0.8)
axes[0][1].set_xticks(x2); axes[0][1].set_xticklabels(cats, rotation=30)
axes[0][1].set_ylim(0,1.1); axes[0][1].set_ylabel('Score')
axes[0][1].set_title(f'Best Model: {best_model_name}\nPrecision / F1 / Recall',
                     fontsize=12, fontweight='bold')
axes[0][1].legend()

# Confusion matrix
cm = confusion_matrix(y_test, predictions[best_model_name], labels=sorted(y.unique()))
ConfusionMatrixDisplay(cm, display_labels=sorted(y.unique())).plot(
    ax=axes[1][0], cmap='Blues', colorbar=False)
axes[1][0].set_title(f'Confusion Matrix: {best_model_name}', fontsize=12, fontweight='bold')
axes[1][0].tick_params(axis='x', rotation=30)

# Macro F1 all models
macro_f1 = [results[n]['report']['macro avg']['f1-score'] for n in names]
bars = axes[1][1].barh(names, macro_f1, color=sns.color_palette("husl", len(names)), alpha=0.85)
for bar, f1 in zip(bars, macro_f1):
    axes[1][1].text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2,
                    f'{f1:.3f}', va='center', fontweight='bold')
axes[1][1].set_xlim(0,1.15); axes[1][1].set_xlabel('Macro F1-Score')
axes[1][1].set_title('Macro F1-Score: All Classifiers', fontsize=13, fontweight='bold')

plt.suptitle('Module 7: Text Classification Results', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()
print(f"✅ Module 7 complete! Best: {best_model_name} ({results[best_model_name]['accuracy']:.1%})")


---
# 🔍 Module 8: Named Entity Recognition and Analysis

NER converts unstructured text into structured data — extracting people, organizations, locations, dates, and monetary values. This is the backbone of media monitoring, financial intelligence, and competitive analysis systems.

In [ ]:
# ============================================================
# CELL 16: NER with spaCy
# ============================================================
ENTITY_TYPES = ['PERSON','ORG','GPE','DATE','MONEY','EVENT','PRODUCT','LOC']

def extract_entities(text, max_chars=800):
    """Extract named entities using spaCy NER pipeline."""
    if not isinstance(text, str):
        return {}
    try:
        doc = nlp(text[:max_chars])
        ents = defaultdict(list)
        for ent in doc.ents:
            if ent.label_ in ENTITY_TYPES:
                ents[ent.label_].append(ent.text.strip())
        return dict(ents)
    except:
        return {}

print("⏳ Running NER...")
df['entities'] = df['content'].apply(extract_entities)
for et in ENTITY_TYPES:
    df[f'n_{et}'] = df['entities'].apply(
        lambda x: len(x.get(et,[])) if isinstance(x,dict) else 0)
print("✅ NER complete!")

ner_cols  = [f'n_{e}' for e in ENTITY_TYPES]
ner_avail = [c for c in ner_cols if c in df.columns]
ner_by_cat = df.groupby('category')[ner_avail].mean().round(2)
print("\n📊 Avg Entity Counts by Category:")
print(ner_by_cat)

# Aggregate all entities
all_entities = defaultdict(Counter)
for _, row in df.iterrows():
    if isinstance(row['entities'], dict):
        for et, elist in row['entities'].items():
            for e in elist:
                all_entities[et][e] += 1

print("\n🏆 Top 5 Entities per Type:")
for et in ['PERSON','ORG','GPE']:
    if all_entities[et]:
        top = all_entities[et].most_common(5)
        print(f"  {et:<8}: {', '.join(f'{e}({c})' for e,c in top)}")


In [ ]:
# ============================================================
# CELL 17: NER Visualization
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Heatmap
if not ner_by_cat.empty:
    norm_ner = ner_by_cat.copy()
    for col in norm_ner.columns:
        m = norm_ner[col].max()
        if m > 0: norm_ner[col] /= m
    sns.heatmap(norm_ner, annot=ner_by_cat, fmt='.2f', cmap='YlGnBu',
                ax=axes[0][0], linewidths=0.5)
    axes[0][0].set_title('Entity Distribution by Category\n(normalized)', fontsize=12, fontweight='bold')
    axes[0][0].set_xticklabels([c.replace('n_','') for c in ner_avail], rotation=30)

# Total entities per category
totals = df.groupby('category')[ner_avail].sum().sum(axis=1).sort_values(ascending=False)
axes[0][1].bar(totals.index, totals.values, color=sns.color_palette("Set2", len(totals)))
axes[0][1].set_title('Total Named Entities per Category', fontsize=12, fontweight='bold')
axes[0][1].set_ylabel('Count'); axes[0][1].tick_params(axis='x', rotation=30)

# Top PERSON
pc = all_entities.get('PERSON', Counter())
if pc:
    tp = pc.most_common(10)
    p, c = zip(*tp)
    axes[1][0].barh(list(p)[::-1], list(c)[::-1], color='#3498db', alpha=0.8)
    axes[1][0].set_title('Top 10 People (PERSON)', fontsize=12, fontweight='bold')
    axes[1][0].set_xlabel('Mentions')
else:
    axes[1][0].text(0.5,0.5,'No PERSON entities found',ha='center',va='center',transform=axes[1][0].transAxes)
    axes[1][0].set_title('Top People (PERSON)', fontsize=12)

# Top ORG
oc = all_entities.get('ORG', Counter())
if oc:
    to = oc.most_common(10)
    o, c2 = zip(*to)
    axes[1][1].barh(list(o)[::-1], list(c2)[::-1], color='#e67e22', alpha=0.8)
    axes[1][1].set_title('Top 10 Organizations (ORG)', fontsize=12, fontweight='bold')
    axes[1][1].set_xlabel('Mentions')
else:
    axes[1][1].text(0.5,0.5,'No ORG entities found',ha='center',va='center',transform=axes[1][1].transAxes)
    axes[1][1].set_title('Top Organizations (ORG)', fontsize=12)

plt.suptitle('Module 8: Named Entity Recognition Analysis', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()
print("✅ Module 8 complete!")


---
# 🎯 Integration: Full Article Profiler & Executive Dashboard

This section demonstrates all 8 modules working together as a unified pipeline — the core value proposition of the NewsBot system.

In [ ]:
# ============================================================
# CELL 18: Full Article Profiler
# ============================================================
def analyze_article_full(text, true_category=None):
    """Run a single article through all 8 modules and print a full report."""
    print("=" * 62)
    print("🤖 NEWSBOT — FULL ARTICLE ANALYSIS")
    print("=" * 62)
    print(f"Article: {text[:90]}...")

    # M2
    tokens = preprocess_text(text)
    sents  = sent_tokenize(text) if isinstance(text, str) else []
    print(f"\n[M2] Preprocessing:  {len(text.split())} words → {len(tokens)} tokens | {len(sents)} sentences")

    # M3
    print(f"[M3] Key Terms:      {', '.join(tokens[:7])}")

    # M4
    pd_ = get_pos_distribution(text)
    top_pos = sorted(pd_.items(), key=lambda x:-x[1])[:3]
    print(f"[M4] Top POS:        {', '.join(f'{p}({v:.0f}%)' for p,v in top_pos)}")

    # M5
    syn = extract_syntactic_features(text)
    svo = syn.get('svo_triples',[])
    print(f"[M5] Syntax:         avg_sent_len={syn.get('avg_sentence_length',0):.1f} | noun_chunks={syn.get('n_noun_chunks',0)}")
    if svo: print(f"     SVO:           {svo[0]}")

    # M6
    sent = analyze_sentiment(text)
    emoji = {'positive':'😊','negative':'😞','neutral':'😐'}.get(sent['label'],'')
    print(f"[M6] Sentiment:      {emoji} {sent['label'].upper()} (compound={sent['compound']:+.3f} | subjectivity={sent['subjectivity']:.2f})")

    # M7
    best_pipe = classifiers[best_model_name]
    pred_cat  = best_pipe.predict([' '.join(tokens)])[0]
    match = ''
    if true_category:
        match = ' ✅' if pred_cat == true_category else f' ❌ (actual: {true_category})'
    print(f"[M7] Classification: {pred_cat.upper()}{match}")

    # M8
    ents = extract_entities(text)
    print(f"[M8] Entities:")
    if ents:
        for et, elist in list(ents.items())[:4]:
            print(f"     {et:<8}: {', '.join(list(set(elist))[:3])}")
    else:
        print("     None detected (try with longer text)")

    print("=" * 62)
    return pred_cat

# Run on 3 sample articles
for i in [0, 5, 12]:
    row = df.iloc[i]
    analyze_article_full(row['content'], row['category'])
    print()


In [ ]:
# ============================================================
# CELL 19: Executive Intelligence Dashboard
# ============================================================
print("=" * 65)
print("  📊 NEWSBOT INTELLIGENCE SYSTEM — EXECUTIVE DASHBOARD")
print("=" * 65)

print(f"\n🗂️  CORPUS OVERVIEW")
print(f"   Articles analyzed : {len(df)}")
print(f"   Categories        : {', '.join(sorted(df['category'].unique()))}")
print(f"   Avg article length: {df['token_count'].mean():.0f} tokens")
print(f"   Total sentences   : {df['sentence_count'].sum():.0f}")

print(f"\n🤖 CLASSIFICATION LEADERBOARD")
for i,(name,res) in enumerate(sorted(results.items(), key=lambda x:-x[1]['accuracy'])):
    medal = ['🥇','🥈','🥉','  '][min(i,3)]
    f1 = res['report']['macro avg']['f1-score']
    print(f"   {medal} {name:<22} Acc={res['accuracy']:.1%}  F1={f1:.1%}  CV={res['cv_mean']:.1%}±{res['cv_std']:.1%}")

print(f"\n😊 SENTIMENT INTELLIGENCE")
sbc = df.groupby('category')[['sentiment_compound','subjectivity']].mean()
for cat, row in sbc.iterrows():
    tone = 'POSITIVE' if row['sentiment_compound']>0.05 else ('NEGATIVE' if row['sentiment_compound']<-0.05 else 'NEUTRAL')
    print(f"   {cat:<15} {tone:<9} score={row['sentiment_compound']:+.3f}  subj={row['subjectivity']:.2f}")

print(f"\n🔍 TOP NAMED ENTITIES")
for et in ['PERSON','ORG','GPE','MONEY']:
    if all_entities[et]:
        top3 = [e[0] for e in all_entities[et].most_common(3)]
        print(f"   {et:<8}: {', '.join(top3)}")

print(f"\n💡 KEY BUSINESS INSIGHTS")
most_pos = df.groupby('category')['sentiment_compound'].mean().idxmax()
most_neg = df.groupby('category')['sentiment_compound'].mean().idxmin()
longest  = df.groupby('category')['token_count'].mean().idxmax()
most_ent = df.groupby('category')[ner_avail].sum().sum(axis=1).idxmax() if ner_avail else 'N/A'
most_subj= df.groupby('category')['subjectivity'].mean().idxmax()

insights = [
    f"Most positive news coverage  → {most_pos.upper()}",
    f"Most negative news coverage  → {most_neg.upper()}",
    f"Most in-depth articles       → {longest.upper()} (longest avg length)",
    f"Most entity-rich category    → {str(most_ent).upper()}",
    f"Most opinionated writing     → {most_subj.upper()} (highest subjectivity)",
    f"Best classification model    → {best_model_name} ({results[best_model_name]['accuracy']:.1%} accuracy)",
]
for i, insight in enumerate(insights, 1):
    print(f"   {i}. {insight}")

print(f"\n✅ Analysis complete — ready for GitHub submission!")


---
# 📝 Reflection Questions

**Instructions:** Answer each question in your own words (3–5 sentences). These responses demonstrate your understanding of the system and account for a significant portion of your grade. Write from your own observations of the results above.

---

### Module 1 — Business Application
**Q: Who are the primary users of your NewsBot system and what specific value does it deliver to each?**

**Your Answer:**
*[Write here. Example topics: which industries benefit, what decisions does the system enable, how does it save time or money]*

---

### Module 2 — Preprocessing
**Q: Which preprocessing step had the most noticeable impact on your data quality and why?**

**Your Answer:**
*[Write here. Look at the before/after output in Cell 4 and comment on what changed most significantly]*

---

### Module 3 — TF-IDF
**Q: Which news category had the most distinctive vocabulary in your TF-IDF results? What does this tell us about how that category uses language?**

**Your Answer:**
*[Write here. Look at the word clouds and bar charts — which category's top words are most unique and informative?]*

---

### Module 4 — POS Tagging
**Q: What grammatical pattern surprised you most when comparing categories? Why might that pattern exist?**

**Your Answer:**
*[Write here. Look at the heatmap — which category has an unusually high or low rate of a particular POS tag?]*

---

### Module 5 — Syntax Parsing
**Q: How does sentence complexity differ across your news categories? What does that suggest about the writing style of each category?**

**Your Answer:**
*[Write here. Reference the avg sentence length and noun chunk count from your results]*

---

### Module 6 — Sentiment
**Q: Which category had the most negative overall sentiment and does that match your intuition? Explain why the result makes sense (or doesn't).**

**Your Answer:**
*[Write here. Reference the compound scores from the dashboard output]*

---

### Module 7 — Classification
**Q: Which classifier performed best on your dataset and why do you think it outperformed the others for this specific task?**

**Your Answer:**
*[Write here. Explain the winning model's strengths — e.g., why does SVM often outperform Naive Bayes on text?]*

---

### Module 8 — NER
**Q: What patterns did you notice in named entity distribution across categories? Give at least two specific examples from your results.**

**Your Answer:**
*[Write here. E.g., "sport articles had many PERSON entities (athlete names) while business articles had many ORG entities (company names)"]*

---

### Overall Integration
**Q: How does combining all 8 NLP modules create more business value than any single technique could provide alone? Give a concrete example.**

**Your Answer:**
*[Write here. Think about how classification + sentiment + NER together enable richer insights than any one alone — e.g., "knowing an article is about politics AND is highly negative AND mentions a specific politician's name is far more actionable than just knowing it's political"]*

---


In [ ]:
# ============================================================
# CELL 20: Submission Checklist & GitHub Instructions
# ============================================================
print("=" * 62)
print("  📋 SUBMISSION CHECKLIST")
print("=" * 62)

checklist = [
    ("Module 1",    "Business context documented",                 True),
    ("Module 2",    "Preprocessing pipeline complete",             'tokens' in df.columns),
    ("Module 3",    "TF-IDF vectorization + visualization",        'X_tfidf' in dir()),
    ("Module 4",    "POS tagging + heatmap",                       'pos_dist' in df.columns),
    ("Module 5",    "Dependency parsing + SVO triples",            'syntax' in df.columns),
    ("Module 6",    "Sentiment analysis + 4 charts",               'sentiment_label' in df.columns),
    ("Module 7",    "4 classifiers trained & compared",            len(results) >= 3),
    ("Module 8",    "NER extraction + entity charts",              'entities' in df.columns),
    ("Integration", "Full article profiler works",                 True),
    ("Dashboard",   "Executive summary generated",                 True),
    ("Reflection",  "All 9 questions answered (manual)",           False),
    ("Real Data",   "BBC/Kaggle data loaded (replace demo)",       False),
    ("GitHub",      "Repo pushed, public, folder created",         False),
    ("Naming",      "File named MT_Report_..._ITAI2373.ipynb",     False),
    ("Journal",     "Reflective journal PDF written",              False),
]

auto_pass = sum(1 for _,_,d in checklist if d)
total     = len(checklist)

for module, task, done in checklist:
    icon = "✅" if done else "⬜"
    print(f"  {icon} [{module:<12}] {task}")

print(f"\n  Automated checks: {auto_pass}/{total} passed")
print()
print("=" * 62)
print("  📁 GITHUB SUBMISSION STEPS")
print("=" * 62)
print()
print("  1. Download this notebook from Colab:")
print("     File → Download → Download .ipynb")
print()
print("  2. Rename the file to:")
print("     MT_Report_[GroupName]_[YourFullName]_ITAI2373.ipynb")
print()
print("  3. In your GitHub portfolio repo, create folder:")
print("     ITAI2373-NewsBot-Midterm/")
print()
print("  4. Upload notebook + a README.md to that folder")
print()
print("  5. Make your GitHub repo PUBLIC")
print("     Settings → Danger Zone → Change visibility → Public")
print()
print("  6. Submit on Canvas:")
print("     - Your GitHub repository URL")
print("     - Reflective Journal PDF (1 file, 2 pages)")
print()
print("  ⚠️  Test your GitHub link in an incognito window")
print("     before submitting to confirm it's accessible!")
